In [58]:
import sqlite3
import pandas as pd
import numpy as np

In [59]:
import sqlite3

conn = sqlite3.connect(
    r"C:\Users\adeeb\Downloads\archive (5)\travel.sqlite"
)

Q12. Which aircraft types operate the most flights, and which carry the most passengers per flight?

In [60]:
aircraft_flights = pd.read_sql_query("""
    SELECT
        aircraft_code,
        COUNT(*) AS total_flights
    FROM flights
    GROUP BY aircraft_code
""", conn)

aircraft_flights

,aircraft_code,total_flights
0,319,1239
1,321,1952
2,733,1274
3,763,1221
4,773,610
5,CN1,9273
6,CR2,9048
7,SU9,8504


In [61]:
aircraft_passengers = pd.read_sql_query("""
    SELECT
        f.aircraft_code,
        COUNT(tf.ticket_no) AS total_passengers
    FROM flights AS f
    JOIN ticket_flights AS tf
        ON f.flight_id = tf.flight_id
    GROUP BY f.aircraft_code
""", conn)

aircraft_passengers

,aircraft_code,total_passengers
0,319,52853
1,321,107129
2,733,86102
3,763,124774
4,773,144376
5,CN1,14672
6,CR2,150122
7,SU9,365698


In [62]:
aircraft_analysis = aircraft_flights.merge(
    aircraft_passengers,
    on="aircraft_code",
    how="left"
)

aircraft_analysis["passengers_per_flight"] = (
    aircraft_analysis["total_passengers"]
    / aircraft_analysis["total_flights"]
)

aircraft_analysis.sort_values(
    "passengers_per_flight",
    ascending=False
)
aircraft_analysis

,aircraft_code,total_flights,total_passengers,passengers_per_flight
0,319,1239,52853,42.657789
1,321,1952,107129,54.881660
2,733,1274,86102,67.583987
3,763,1221,124774,102.190008
4,773,610,144376,236.681967
5,CN1,9273,14672,1.582228
6,CR2,9048,150122,16.591733
7,SU9,8504,365698,43.003057


Q13:

Which aircraft types and routes have unusually low passenger utilization relative to available seat capacity?

In [63]:
aircraft_capacity = pd.read_sql_query("""
    SELECT
        aircraft_code,
        COUNT(seat_no) AS total_seats
    FROM seats
    GROUP BY aircraft_code
""", conn)

aircraft_capacity

,aircraft_code,total_seats
0,319,116
1,320,140
2,321,170
3,733,130
4,763,222
5,773,402
6,CN1,12
7,CR2,50
8,SU9,97


In [64]:
aircraft_analysis = aircraft_analysis.merge(
    aircraft_capacity,
    on="aircraft_code",
    how="left"
)

aircraft_analysis

,aircraft_code,total_flights,total_passengers,passengers_per_flight,total_seats
0,319,1239,52853,42.657789,116
1,321,1952,107129,54.881660,170
2,733,1274,86102,67.583987,130
3,763,1221,124774,102.190008,222
4,773,610,144376,236.681967,402
5,CN1,9273,14672,1.582228,12
6,CR2,9048,150122,16.591733,50
7,SU9,8504,365698,43.003057,97


In [65]:
aircraft_analysis["utilization"] = (
    aircraft_analysis["total_passengers"]
    /
    (
        aircraft_analysis["total_flights"]
        * aircraft_analysis["total_seats"]
    )
    * 100
)

In [66]:
aircraft_analysis.sort_values(
    "utilization",
    ascending=True
)

,aircraft_code,total_flights,total_passengers,passengers_per_flight,total_seats,utilization
5,CN1,9273,14672,1.582228,12,13.185233
1,321,1952,107129,54.881660,170,32.283329
6,CR2,9048,150122,16.591733,50,33.183466
0,319,1239,52853,42.657789,116,36.773956
7,SU9,8504,365698,43.003057,97,44.333049
3,763,1221,124774,102.190008,222,46.031535
2,733,1274,86102,67.583987,130,51.987683
4,773,610,144376,236.681967,402,58.876111


Aircraft utilization varies substantially across the fleet. The Cessna 208 has the lowest utilization at only 13.2%, followed by the A321 (32.3%) and CRJ-200 (33.2%). In contrast, the Boeing 777-300 achieves the highest utilization at 58.9%. This suggests that regional/smaller aircraft have substantially more unused capacity, while larger aircraft generally operate with higher passenger load factors.

In [67]:
flight_passengers = pd.read_sql_query("""
    SELECT
        flight_id,
        COUNT(ticket_no) AS passengers
    FROM ticket_flights
    GROUP BY flight_id
""", conn)

flight_passengers.head()

,flight_id,passengers
0,1,79
1,2,101
2,3,97
3,5,93
4,6,101


In [68]:
flight_details = pd.read_sql_query("""
    SELECT
        flight_id,
        departure_airport,
        arrival_airport,
        aircraft_code
    FROM flights
""", conn)

flight_details.head()

,flight_id,departure_airport,arrival_airport,aircraft_code
0,1185,DME,BTK,319
1,3979,VKO,HMA,CR2
2,4739,VKO,AER,763
3,5502,SVO,UFA,763
4,6938,SVO,ULV,SU9


In [69]:
route_flights = flight_details.merge(
    flight_passengers,
    on="flight_id",
    how="left"
)

route_flights.head()

,flight_id,departure_airport,arrival_airport,aircraft_code,passengers
0,1185,DME,BTK,319,2.0
1,3979,VKO,HMA,CR2,28.0
2,4739,VKO,AER,763,41.0
3,5502,SVO,UFA,763,9.0
4,6938,SVO,ULV,SU9,15.0


In [70]:
route_flights = route_flights.merge(
    aircraft_capacity,
    on="aircraft_code",
    how="left"
)

route_flights.head()

,flight_id,departure_airport,arrival_airport,aircraft_code,passengers,total_seats
0,1185,DME,BTK,319,2.0,116
1,3979,VKO,HMA,CR2,28.0,50
2,4739,VKO,AER,763,41.0,222
3,5502,SVO,UFA,763,9.0,222
4,6938,SVO,ULV,SU9,15.0,97


In [71]:
route_flights["utilization"] = (
    route_flights["passengers"]
    / route_flights["total_seats"]
    * 100
)

route_flights.head()

,flight_id,departure_airport,arrival_airport,aircraft_code,passengers,total_seats,utilization
0,1185,DME,BTK,319,2.0,116,1.724138
1,3979,VKO,HMA,CR2,28.0,50,56.000000
2,4739,VKO,AER,763,41.0,222,18.468468
3,5502,SVO,UFA,763,9.0,222,4.054054
4,6938,SVO,ULV,SU9,15.0,97,15.463918


In [72]:
route_flights["route"] = (
    route_flights[["departure_airport", "arrival_airport"]]
    .apply(lambda x: " ↔ ".join(sorted(x)), axis=1)
)

In [73]:
route_flights[
    ["departure_airport", "arrival_airport", "route"]
].head()

,departure_airport,arrival_airport,route
0,DME,BTK,BTK ↔ DME
1,VKO,HMA,HMA ↔ VKO
2,VKO,AER,AER ↔ VKO
3,SVO,UFA,SVO ↔ UFA
4,SVO,ULV,SVO ↔ ULV


In [74]:
route_flights

,flight_id,departure_airport,arrival_airport,aircraft_code,passengers,total_seats,utilization,route
0,1185,DME,BTK,319,2.0,116,1.724138,BTK ↔ DME
1,3979,VKO,HMA,CR2,28.0,50,56.000000,HMA ↔ VKO
2,4739,VKO,AER,763,41.0,222,18.468468,AER ↔ VKO
3,5502,SVO,UFA,763,9.0,222,4.054054,SVO ↔ UFA
4,6938,SVO,ULV,SU9,15.0,97,15.463918,SVO ↔ ULV
...,...,...,...,...,...,...,...,...
33116,33117,SKX,SVO,CR2,16.0,50,32.000000,SKX ↔ SVO
33117,33118,SKX,SVO,CR2,16.0,50,32.000000,SKX ↔ SVO
33118,33119,SKX,SVO,CR2,4.0,50,8.000000,SKX ↔ SVO
33119,33120,SKX,SVO,CR2,13.0,50,26.000000,SKX ↔ SVO


In [75]:
route_utilization = (
    route_flights
    .groupby("route")
    .agg(
        total_flights=("flight_id", "count"),
        total_passengers=("passengers", "sum"),
        total_available_seats=("total_seats", "sum")
    )
    .reset_index()
)
route_utilization

,route,total_flights,total_passengers,total_available_seats
0,AAQ ↔ EGO,122,8610.0,11834
1,AAQ ↔ NOZ,18,0.0,2340
2,AAQ ↔ SVO,122,10626.0,15860
3,ABA ↔ ARH,16,0.0,1856
4,ABA ↔ DME,35,1955.0,4060
...,...,...,...,...
304,UUD ↔ VKO,52,737.0,6032
305,UUD ↔ YKS,36,0.0,1800
306,UUS ↔ YKS,36,0.0,1800
307,VKO ↔ VOG,122,7101.0,11834


In [76]:
route_utilization["utilization"] = (
    route_utilization["total_passengers"]
    / route_utilization["total_available_seats"]
    * 100
)

In [77]:
route_utilization.sort_values(
    "utilization",
    ascending=True
).head(15)

,route,total_flights,total_passengers,total_available_seats,utilization
151,KEJ ↔ KYZ,35,0.0,420,0.0
233,NOZ ↔ TOF,122,0.0,1464,0.0
234,NSK ↔ NYM,35,0.0,420,0.0
68,DME ↔ JOK,122,0.0,1464,0.0
237,NYA ↔ ROV,17,0.0,850,0.0
64,DME ↔ EYK,18,0.0,900,0.0
239,NYA ↔ VKO,18,0.0,900,0.0
121,GDZ ↔ ROV,35,0.0,420,0.0
61,DME ↔ DYR,18,0.0,2088,0.0
231,NOZ ↔ NYM,35,0.0,1750,0.0


In [78]:
route_utilization_positive = route_utilization[
    route_utilization["total_passengers"] > 0
].sort_values(
    "utilization",
    ascending=True
)

route_utilization_positive.head(15)


,route,total_flights,total_passengers,total_available_seats,utilization
211,MJZ ↔ SVX,34,11.0,3298,0.333535
196,LED ↔ OVS,122,76.0,6100,1.245902
143,IKT ↔ MJZ,122,97.0,6100,1.590164
257,OVS ↔ UFA,122,177.0,6100,2.901639
50,CEK ↔ SWT,122,44.0,1464,3.005464
299,TJM ↔ UCT,122,227.0,6100,3.721311
193,KZN ↔ ROV,244,140.0,2928,4.781421
164,KJA ↔ NOZ,244,188.0,2928,6.420765
43,CEE ↔ LED,244,209.0,2928,7.137978
271,RGK ↔ SVO,18,183.0,2340,7.820513


Aircraft-level utilization ranges from just 13.2% for the Cessna 208 to 58.9% for the Boeing 777-300, indicating substantial variation in capacity utilization across the fleet. At the route level, several routes show severe under-utilization despite having actual passenger demand. MJZ–SVX has the lowest utilization at only 0.33%, followed by LED–OVS (1.25%), IKT–MJZ (1.59%), OVS–UFA (2.90%), and CEK–SWT (3.01%). These routes indicate significant mismatches between scheduled capacity and passenger demand and may warrant review of aircraft assignment, flight frequency, or route viability.

Q14. Which airports and routes experience the highest concentration of flight activity?
Operational Reliability


In [81]:
flights = pd.read_sql_query("""select * from flights """ , conn)

In [82]:
airport_activity = pd.concat([
    flights["departure_airport"],
    flights["arrival_airport"]
]).value_counts().reset_index()

airport_activity.columns = ["airport", "total_flights"]

airport_activity.head(15)
airport_activity

,airport,total_flights
0,DME,6434
1,SVO,5963
2,LED,3802
3,VKO,3436
4,OVB,2110
...,...,...
99,PYJ,54
100,NYA,53
101,PKC,52
102,USK,36


In [83]:
#Route activity
route_flights["route"] = route_flights[
    ["departure_airport", "arrival_airport"]
].apply(
    lambda x: " ↔ ".join(sorted(x)),
    axis=1
)

route_flights.head()

,flight_id,departure_airport,arrival_airport,aircraft_code,passengers,total_seats,utilization,route
0,1185,DME,BTK,319,2.0,116,1.724138,BTK ↔ DME
1,3979,VKO,HMA,CR2,28.0,50,56.000000,HMA ↔ VKO
2,4739,VKO,AER,763,41.0,222,18.468468,AER ↔ VKO
3,5502,SVO,UFA,763,9.0,222,4.054054,SVO ↔ UFA
4,6938,SVO,ULV,SU9,15.0,97,15.463918,SVO ↔ ULV


In [84]:
route_activity = (
    route_flights["route"]
    .value_counts()
    .reset_index()
)

route_activity.columns = ["route", "total_flights"]

route_activity.head(15)

,route,total_flights
0,LED ↔ SVO,610
1,DME ↔ LED,488
2,BZK ↔ SVO,366
3,BZK ↔ DME,366
4,BZK ↔ VKO,366
5,LED ↔ VKO,366
6,NOZ ↔ OVB,244
7,ARH ↔ PEE,244
8,HMA ↔ NUX,244
9,KGP ↔ SVX,244


Flight activity is highly concentrated around the major airports, with DME handling the most airport-level activity (6,434), followed by SVO (5,963), LED (3,802), and VKO (3,436). At the route level, LED ↔ SVO is the busiest route with 610 flights, followed by DME ↔ LED with 488 flights. Several other routes record 366 flights each, indicating a smaller but still significant concentration of activity.

**Q15. Which routes and aircraft types have the highest delay and cancellation rates?**

In [85]:
flights["status"].value_counts()

status
Arrived      16707
Scheduled    15383
On Time        518
Cancelled      414
Departed        58
Delayed         41
Name: count, dtype: int64

In [86]:
flights["route"] = flights[
    ["departure_airport", "arrival_airport"]
].apply(
    lambda x: " ↔ ".join(sorted(x)),
    axis=1
)

In [87]:
route_reliability = (
    flights.groupby("route")
    .agg(
        total_flights=("flight_id", "count"),
        delayed_flights=("status", lambda x: (x == "Delayed").sum()),
        cancelled_flights=("status", lambda x: (x == "Cancelled").sum())
    )
    .reset_index()
)

route_reliability["delay_rate"] = (
    route_reliability["delayed_flights"]
    / route_reliability["total_flights"] * 100
)

route_reliability["cancellation_rate"] = (
    route_reliability["cancelled_flights"]
    / route_reliability["total_flights"] * 100
)

In [88]:
route_reliability_100 = route_reliability[
    route_reliability["total_flights"] >= 100   #To avoid unstable rates from routes with very few flights, I restricted reliability comparisons to routes with at least 100 flights."
]

In [89]:
route_reliability_100.sort_values(
    "delay_rate",
    ascending=False
).head(15)

,route,total_flights,delayed_flights,cancelled_flights,delay_rate,cancellation_rate
182,KVX ↔ KZN,122,1,2,0.819672,1.639344
168,KJA ↔ SVO,122,1,1,0.819672,0.819672
75,DME ↔ KVX,122,1,2,0.819672,1.639344
134,HMA ↔ VKO,122,1,2,0.819672,1.639344
81,DME ↔ NAL,122,1,2,0.819672,1.639344
212,MJZ ↔ YKS,122,1,1,0.819672,0.819672
266,PKV ↔ SVO,122,1,1,0.819672,0.819672
141,IKT ↔ KZN,122,1,1,0.819672,0.819672
142,IKT ↔ LED,122,1,1,0.819672,0.819672
145,IKT ↔ SGC,122,1,1,0.819672,0.819672


In [90]:
route_reliability_100.sort_values(
    "cancellation_rate",
    ascending=False
).head(15)

,route,total_flights,delayed_flights,cancelled_flights,delay_rate,cancellation_rate
163,KJA ↔ KRO,122,0,2,0.000000,1.639344
216,MQF ↔ SVX,122,0,2,0.000000,1.639344
114,ESL ↔ SVO,122,0,2,0.000000,1.639344
113,ESL ↔ REN,122,0,2,0.000000,1.639344
109,EGO ↔ ROV,122,0,2,0.000000,1.639344
105,DME ↔ VOZ,122,0,2,0.000000,1.639344
104,DME ↔ VKT,122,1,2,0.819672,1.639344
103,DME ↔ UUS,122,0,2,0.000000,1.639344
102,DME ↔ UUA,122,0,2,0.000000,1.639344
224,NJC ↔ OVS,122,0,2,0.000000,1.639344


Route-level operational disruptions are generally low. Among routes with at least 100 flights, the highest cancellation rate is 1.64%, occurring on multiple routes with 2 cancellations out of 122 flights. Delay rates are similarly low, with the highest rate among these routes at 0.82%.

In [91]:
aircraft_reliability = (
    flights.groupby("aircraft_code")
    .agg(
        total_flights=("flight_id", "count"),
        delayed_flights=("status", lambda x: (x == "Delayed").sum()),
        cancelled_flights=("status", lambda x: (x == "Cancelled").sum())
    )
    .reset_index()
)

aircraft_reliability["delay_rate"] = (
    aircraft_reliability["delayed_flights"]
    / aircraft_reliability["total_flights"] * 100
)

aircraft_reliability["cancellation_rate"] = (
    aircraft_reliability["cancelled_flights"]
    / aircraft_reliability["total_flights"] * 100
)
aircraft_reliability

,aircraft_code,total_flights,delayed_flights,cancelled_flights,delay_rate,cancellation_rate
0,319,1239,2,24,0.161421,1.937046
1,321,1952,4,3,0.204918,0.153689
2,733,1274,1,15,0.078493,1.177394
3,763,1221,1,6,0.081900,0.491400
4,773,610,0,1,0.000000,0.163934
5,CN1,9273,12,115,0.129408,1.240160
6,CR2,9048,11,181,0.121574,2.000442
7,SU9,8504,10,69,0.117592,0.811383


In [92]:
aircraft_reliability.sort_values(
    "cancellation_rate",
    ascending=False
)

,aircraft_code,total_flights,delayed_flights,cancelled_flights,delay_rate,cancellation_rate
6,CR2,9048,11,181,0.121574,2.000442
0,319,1239,2,24,0.161421,1.937046
5,CN1,9273,12,115,0.129408,1.240160
2,733,1274,1,15,0.078493,1.177394
7,SU9,8504,10,69,0.117592,0.811383
3,763,1221,1,6,0.081900,0.491400
4,773,610,0,1,0.000000,0.163934
1,321,1952,4,3,0.204918,0.153689


Operational reliability varies more noticeably by aircraft type than by high-volume route. The Bombardier CRJ-200 (CR2) has the highest cancellation rate at 2.00%, followed by the Airbus A319-100 (319) at 1.94%. The A321-200 (321) has the highest delay rate at 0.205%, although its cancellation rate is only 0.154%. At the route level, disruption rates are considerably lower: among routes with at least 100 flights, the highest delay rate is 0.82% and the highest cancellation rate is 1.64%.

Q16. Which high-demand routes have both strong passenger demand and poor operational reliability?

In [286]:
route_demand_reliability = route_utilization.merge(
    route_reliability,
    on="route",
    how="inner"
)

In [287]:

route_demand_reliability = route_demand_reliability.drop(columns=["total_flights_y"])
route_demand_reliability = route_demand_reliability.rename(columns={"total_flights_x": "total_flights"})

In [288]:
route_demand_reliability.head()

,route,total_flights,total_passengers,total_available_seats,utilization,delayed_flights,cancelled_flights,delay_rate,cancellation_rate
0,AAQ ↔ EGO,122,8610.0,11834,72.756464,0,1,0.0,0.819672
1,AAQ ↔ NOZ,18,0.0,2340,0.000000,0,0,0.0,0.000000
2,AAQ ↔ SVO,122,10626.0,15860,66.998739,0,1,0.0,0.819672
3,ABA ↔ ARH,16,0.0,1856,0.000000,0,2,0.0,12.500000
4,ABA ↔ DME,35,1955.0,4060,48.152709,0,0,0.0,0.000000


In [290]:
route_reliability_filtered = route_reliability[route_reliability["total_flights"] >= 100].copy()

route_reliability_filtered.to_csv("route_reliability_filtered.csv", index=False)
route_reliability_filtered

,route,total_flights,delayed_flights,cancelled_flights,delay_rate,cancellation_rate
0,AAQ ↔ EGO,122,0,1,0.000000,0.819672
2,AAQ ↔ SVO,122,0,1,0.000000,0.819672
7,ABA ↔ OVB,244,1,2,0.409836,0.819672
8,ABA ↔ TOF,122,0,2,0.000000,1.639344
9,AER ↔ EGO,244,1,2,0.409836,0.819672
...,...,...,...,...,...,...
300,TJM ↔ URJ,122,0,1,0.000000,0.819672
302,UCT ↔ UFA,244,0,2,0.000000,0.819672
303,ULV ↔ VKO,122,0,1,0.000000,0.819672
307,VKO ↔ VOG,122,0,1,0.000000,0.819672


In [107]:
high_demand_threshold = route_demand_reliability["total_passengers"].quantile(0.75)

high_demand_threshold

np.float64(4629.0)

In [108]:
reliability_threshold = (
    route_demand_reliability["delay_rate"]
    + route_demand_reliability["cancellation_rate"]
).quantile(0.75)

reliability_threshold

np.float64(3.7735849056603774)

In [109]:
high_demand_unreliable = route_demand_reliability[
    (route_demand_reliability["total_passengers"] >= high_demand_threshold) &
    (
        route_demand_reliability["delay_rate"]
        + route_demand_reliability["cancellation_rate"]
        >= reliability_threshold
    )
]

In [110]:
high_demand_unreliable = high_demand_unreliable.sort_values(
    "total_passengers",
    ascending=False
)

high_demand_unreliable

,route,total_flights_x,total_passengers,total_available_seats,utilization,total_flights_y,delayed_flights,cancelled_flights,delay_rate,cancellation_rate


In [111]:
high_demand_routes = route_demand_reliability[
    route_demand_reliability["total_passengers"] >= 4629
].sort_values(
    "total_passengers",
    ascending=False
)

high_demand_routes[
    [
        "route",
        "total_passengers",
        "delay_rate",
        "cancellation_rate"
    ]
]

,route,total_passengers,delay_rate,cancellation_rate
199,LED ↔ SVO,31885.0,0.163934,0.000000
287,SVO ↔ SVX,31312.0,0.000000,0.000000
87,DME ↔ OVB,31305.0,0.000000,0.000000
19,AER ↔ SVO,29816.0,0.000000,0.000000
253,OVB ↔ SVO,25997.0,0.000000,0.819672
...,...,...,...,...
255,OVS ↔ SVO,4895.0,0.000000,1.639344
303,ULV ↔ VKO,4753.0,0.000000,0.819672
243,OMS ↔ SVO,4746.0,0.000000,0.819672
236,NUX ↔ SVO,4701.0,0.000000,0.819672


In [112]:
high_demand_unreliable = high_demand_routes[
    (high_demand_routes["delay_rate"] > 0) |
    (high_demand_routes["cancellation_rate"] > 0)
].sort_values(
    "total_passengers",
    ascending=False
)

high_demand_unreliable[
    [
        "route",
        "total_passengers",
        "delay_rate",
        "cancellation_rate"
    ]
]

,route,total_passengers,delay_rate,cancellation_rate
199,LED ↔ SVO,31885.0,0.163934,0.000000
253,OVB ↔ SVO,25997.0,0.000000,0.819672
202,LED ↔ VKO,21735.0,0.000000,0.273224
77,DME ↔ LED,19382.0,0.204918,0.000000
76,DME ↔ KZN,14202.0,0.000000,0.819672
142,IKT ↔ LED,13759.0,0.819672,0.819672
272,ROV ↔ SVO,11622.0,0.000000,0.819672
15,AER ↔ KUF,11614.0,0.819672,0.000000
208,MCX ↔ SVO,11381.0,0.000000,0.819672
2,AAQ ↔ SVO,10626.0,0.000000,0.819672


No major high-demand routes show genuinely poor operational reliability. Most high-demand routes have either zero disruptions or very low delay/cancellation rates. IKT ↔ LED is the most notable high-demand route with both a delay and cancellation rate (0.82% each), while OVB ↔ SVO has a 0.82% cancellation rate. These routes may warrant monitoring, but the data does not support labeling them operationally poor.

Q17 :Which routes are strategically important because they combine high passenger volume with high commercial value?

Network Strategy

In [244]:
route_commercial = (
    ticket_routes.groupby("route")
    .agg(
        total_passengers=("ticket_no", "count"),
        total_revenue=("amount", "sum")
    )
    .reset_index()
)
route_commercial

,route,total_passengers,total_revenue
0,AAQ ↔ EGO,8610,68020200
1,AAQ ↔ SVO,10626,154633600
2,ABA ↔ DME,1955,88060900
3,ABA ↔ OVB,901,5225800
4,ABA ↔ TOF,948,4645200
...,...,...,...
224,UCT ↔ UFA,337,3336300
225,ULV ↔ VKO,4753,43014600
226,UUD ↔ VKO,737,43826400
227,VKO ↔ VOG,7101,80121900


In [245]:
high_revenue_threshold = route_commercial["total_revenue"].quantile(0.75)

high_revenue_threshold

np.float64(88060900.0)

In [246]:

ticket_flights = pd.read_sql(""" select * from ticket_flights """ , conn)
ticket_flights.head()

,ticket_no,flight_id,fare_conditions,amount
0,0005432159776,30625,Business,42100
1,0005435212351,30625,Business,42100
2,0005435212386,30625,Business,42100
3,0005435212381,30625,Business,42100
4,0005432211370,30625,Business,42100


In [247]:
ticket_routes = ticket_flights.merge(
    flights[[
        "flight_id",
        "departure_airport",
        "arrival_airport"
    ]],
    on="flight_id",
    how="left"
)

In [248]:
ticket_routes["route"] = ticket_routes[
    ["departure_airport", "arrival_airport"]
].apply(
    lambda x: " ↔ ".join(sorted(x)),
    axis=1
)

In [249]:
route_commercial = (
    ticket_routes.groupby("route")
    .agg(
        total_passengers=("ticket_no", "count"),
        total_revenue=("amount", "sum")
    )
    .reset_index()
)
route_commercial

,route,total_passengers,total_revenue
0,AAQ ↔ EGO,8610,68020200
1,AAQ ↔ SVO,10626,154633600
2,ABA ↔ DME,1955,88060900
3,ABA ↔ OVB,901,5225800
4,ABA ↔ TOF,948,4645200
...,...,...,...
224,UCT ↔ UFA,337,3336300
225,ULV ↔ VKO,4753,43014600
226,UUD ↔ VKO,737,43826400
227,VKO ↔ VOG,7101,80121900


In [250]:
route_commercial.sort_values(
    "total_revenue",
    ascending=False
).head(15)

,route,total_passengers,total_revenue
53,DME ↔ KHV,19013,1487276100
68,DME ↔ OVB,31305,1079722600
115,KHV ↔ LED,12766,1006423100
183,OVB ↔ SVO,25997,904181600
107,IKT ↔ LED,13759,815653300
209,SVO ↔ SVX,31312,556222600
12,AER ↔ SVO,29816,516307500
106,IKT ↔ KZN,9800,465865200
217,SVO ↔ UUS,5178,458787600
123,KJA ↔ SVO,9266,416379800


In [251]:
high_passenger_threshold = route_commercial["total_passengers"].quantile(0.75)

high_passenger_threshold

np.float64(6423.0)

In [252]:
high_revenue_threshold = route_commercial["total_revenue"].quantile(0.75)

high_revenue_threshold

np.float64(88060900.0)

In [253]:
strategic_routes = route_commercial[
    (route_commercial["total_passengers"] >= high_passenger_threshold) &
    (route_commercial["total_revenue"] >= high_revenue_threshold)
].sort_values(
    "total_revenue",
    ascending=False
)

strategic_routes[
    ["route", "total_passengers", "total_revenue"]
]

,route,total_passengers,total_revenue
53,DME ↔ KHV,19013,1487276100
68,DME ↔ OVB,31305,1079722600
115,KHV ↔ LED,12766,1006423100
183,OVB ↔ SVO,25997,904181600
107,IKT ↔ LED,13759,815653300
209,SVO ↔ SVX,31312,556222600
12,AER ↔ SVO,29816,516307500
106,IKT ↔ KZN,9800,465865200
123,KJA ↔ SVO,9266,416379800
189,PEE ↔ VKO,25946,374771200


Q18. Which high-volume routes generate below-average revenue per passenger and therefore warrant pricing or product investigation?

In [254]:
route_commercial["revenue_per_passenger"] = (
    route_commercial["total_revenue"]
    / route_commercial["total_passengers"]
)

In [255]:
average_revenue_per_passenger = (
    route_commercial["revenue_per_passenger"].mean()
)

average_revenue_per_passenger

np.float64(20016.05599946692)

In [256]:
high_volume_threshold = (
    route_commercial["total_passengers"].quantile(0.75)
)

high_volume_threshold

np.float64(6423.0)

In [257]:
high_volume_low_revenue = route_commercial[
    (route_commercial["total_passengers"] >= high_volume_threshold) &
    (route_commercial["revenue_per_passenger"] < average_revenue_per_passenger)
]

In [258]:
high_volume_low_revenue = high_volume_low_revenue.sort_values(
    "total_passengers",
    ascending=False
)

In [259]:
high_volume_low_revenue[
    [
        "route",
        "total_passengers",
        "total_revenue",
        "revenue_per_passenger"
    ]
]

,route,total_passengers,total_revenue,revenue_per_passenger
142,LED ↔ SVO,31885,254899200,7994.329622
209,SVO ↔ SVX,31312,556222600,17763.879663
12,AER ↔ SVO,29816,516307500,17316.457607
189,PEE ↔ VKO,25946,374771200,14444.276574
145,LED ↔ VKO,21735,182319800,8388.304578
59,DME ↔ LED,19382,172852700,8918.207615
55,DME ↔ KRR,16618,245475300,14771.651222
212,SVO ↔ UFA,15942,238865500,14983.408606
13,AER ↔ VKO,15656,271258900,17326.194430
58,DME ↔ KZN,14202,136811900,9633.284045


The average revenue per passenger is approximately ₹20,016. High-volume routes with substantially below-average revenue per passenger include BZK ↔ SVO (₹4,639), LED ↔ SVO (₹7,994), LED ↔ VKO (₹8,388), and DME ↔ LED (₹8,918). LED ↔ SVO is the most significant route for investigation because it combines the highest passenger volume in this group (31,885) with revenue per passenger about 60% below the overall average. These routes warrant investigation into fare structure, discounts, cabin mix, and product/pricing strategy.

Q19. How concentrated is the airline's network, and what percentage of passengers and flights are handled by the top 10% of routes and airports?

In [260]:
route_activity

,route,total_flights
0,LED ↔ SVO,610
1,DME ↔ LED,488
2,BZK ↔ SVO,366
3,BZK ↔ DME,366
4,BZK ↔ VKO,366
...,...,...
304,KRR ↔ NOZ,17
305,DYR ↔ SVO,17
306,PES ↔ ROV,17
307,ARH ↔ IKT,16


In [261]:
route_concentration = (
    ticket_routes.groupby("route")
    .agg(
        total_passengers=("ticket_no", "count")
    )
    .reset_index()
)

In [262]:
route_concentration = route_concentration.merge(
    route_activity[["route", "total_flights"]],
    on="route",
    how="left"
)
route_concentration.head()

,route,total_passengers,total_flights
0,AAQ ↔ EGO,8610,122
1,AAQ ↔ SVO,10626,122
2,ABA ↔ DME,1955,35
3,ABA ↔ OVB,901,244
4,ABA ↔ TOF,948,122


In [263]:
route_concentration = route_concentration.rename(
    columns={"total_flights_x": "total_flights"}
)

In [264]:
# Find the top 10% of routes
top_10_routes = int(len(route_concentration) * 0.10)


top_routes = route_concentration.nlargest(
    top_10_routes,
    "total_passengers"
)

In [265]:
# Calculate passenger share
route_passenger_share = (
    top_routes["total_passengers"].sum()
    / route_concentration["total_passengers"].sum()
) * 100

In [266]:
route_passenger_share, route_flight_share

(np.float64(39.132526111046296), np.float64(13.514382705020548))

In [267]:
airport_data = pd.concat([
    route_flights[["departure_airport", "passengers"]].rename(
        columns={"departure_airport": "airport"}
    ),
    route_flights[["arrival_airport", "passengers"]].rename(
        columns={"arrival_airport": "airport"}
    )
])

In [268]:
airport_concentration = (
    airport_data.groupby("airport")
    .agg(
        total_passengers=("passengers", "sum"),
        total_flights=("passengers", "count")
    )
    .reset_index()
)

In [269]:
airport_concentration.head()

,airport,total_passengers,total_flights
0,AAQ,19236.0,239
1,ABA,3804.0,261
2,AER,64816.0,698
3,ARH,10398.0,449
4,ASF,5225.0,265


In [270]:
top_10_airports = int(len(airport_concentration) * 0.10)

top_10_airports

10

In [271]:
top_airports = airport_concentration.nlargest(
    top_10_airports,
    "total_passengers"
)

In [272]:
airport_passenger_share = (
    top_airports["total_passengers"].sum()
    / airport_concentration["total_passengers"].sum()
) * 100

In [273]:
airport_flight_share = (
    top_airports["total_flights"].sum()
    / airport_concentration["total_flights"].sum()
) * 100

In [284]:
airport_flight_share = (
    top_airports["total_flights"].sum()
    / airport_concentration["total_flights"].sum()
) * 100

In [285]:
airport_passenger_share, airport_flight_share

(np.float64(58.35916865412163), np.float64(45.934941060019796))


The airline's network is significantly concentrated at its busiest airports. The top 10% of airports handle approximately 58.36% of all passengers and 45.93% of all flight activity. At the route level, the top 10% of routes handle approximately 39.13% of passengers but only 13.51% of flights. This indicates that passenger demand is concentrated among a relatively small number of routes, while airport activity is also heavily concentrated around major hubs.

In [276]:
q19_summary = pd.DataFrame({
    "level": ["Routes", "Airports"],
    "top_10_percent_share_flights": [
        route_flight_share,
        airport_flight_share
    ],
    "top_10_percent_share_passengers": [
        route_passenger_share,
        airport_passenger_share
    ]
})

q19_summary

,level,top_10_percent_share_flights,top_10_percent_share_passengers
0,Routes,13.514383,39.132526
1,Airports,45.934941,58.359169


In [277]:
#Main flight-level table
route_flights.to_csv("route_flights.csv", index=False)

In [278]:
#Commercial route table
route_commercial.to_csv("route_commercial.csv", index=False)

In [279]:
#Route concentration table
route_concentration.to_csv("route_concentration.csv", index=False)

In [280]:
#Airport concentration table
airport_concentration.to_csv("airport_concentration.csv", index=False)

In [281]:
#Q19 summary
q19_summary.to_csv("q19_summary.csv", index=False)

In [152]:
route_reliability.to_csv("route_reliability.csv", index=False)

In [153]:
aircraft_reliability.to_csv("aircraft_reliability.csv", index=False)

In [154]:
# Confirm both columns match before dropping one
print((route_demand_reliability["total_flights_x"] == route_demand_reliability["total_flights_y"]).all())

route_demand_reliability = route_demand_reliability.drop(columns=["total_flights_y"])
route_demand_reliability = route_demand_reliability.rename(columns={"total_flights_x": "total_flights"})
route_demand_reliability.to_csv("route_reliability.csv", index=False)


True


In [155]:
import json

# --- aircraft_capacity_reliability.csv ---
aircraft_reliability = pd.read_sql_query("""
    SELECT
        aircraft_code,
        COUNT(flight_id) AS total_flights,
        SUM(CASE WHEN status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_flights,
        SUM(CASE WHEN status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_flights
    FROM flights
    GROUP BY aircraft_code
""", conn)

aircraft_reliability["delay_rate"] = (
    aircraft_reliability["delayed_flights"] / aircraft_reliability["total_flights"] * 100
)
aircraft_reliability["cancellation_rate"] = (
    aircraft_reliability["cancelled_flights"] / aircraft_reliability["total_flights"] * 100
)

aircraft_capacity = pd.read_sql_query("""
    SELECT aircraft_code, COUNT(seat_no) AS total_seats
    FROM seats
    GROUP BY aircraft_code
""", conn)

aircrafts_data = pd.read_sql_query("SELECT aircraft_code, model FROM aircrafts_data", conn)
aircrafts_data["model_name"] = aircrafts_data["model"].apply(lambda x: json.loads(x)["en"])
aircrafts_data = aircrafts_data[["aircraft_code", "model_name"]]

aircraft_capacity_reliability = (
    aircraft_reliability
    .merge(aircraft_capacity, on="aircraft_code", how="left")
    .merge(aircrafts_data, on="aircraft_code", how="left")
)

aircraft_capacity_reliability = aircraft_capacity_reliability[[
    "aircraft_code", "model_name", "total_seats", "total_flights",
    "delayed_flights", "cancelled_flights", "delay_rate", "cancellation_rate"
]].sort_values("total_flights", ascending=False)

aircraft_capacity_reliability.to_csv("aircraft_capacity_reliability.csv", index=False)
aircraft_capacity_reliability

,aircraft_code,model_name,total_seats,total_flights,delayed_flights,cancelled_flights,delay_rate,cancellation_rate
5,CN1,Cessna 208 Caravan,12,9273,12,115,0.129408,1.240160
6,CR2,Bombardier CRJ-200,50,9048,11,181,0.121574,2.000442
7,SU9,Sukhoi Superjet-100,97,8504,10,69,0.117592,0.811383
1,321,Airbus A321-200,170,1952,4,3,0.204918,0.153689
2,733,Boeing 737-300,130,1274,1,15,0.078493,1.177394
0,319,Airbus A319-100,116,1239,2,24,0.161421,1.937046
3,763,Boeing 767-300,222,1221,1,6,0.081900,0.491400
4,773,Boeing 777-300,402,610,0,1,0.000000,0.163934


In [156]:
# --- flight_status_summary.csv ---
flight_status_summary = pd.read_sql_query("""
    SELECT status, COUNT(flight_id) AS total_flights
    FROM flights
    GROUP BY status
""", conn)

flight_status_summary["pct_of_total"] = (
    flight_status_summary["total_flights"] / flight_status_summary["total_flights"].sum() * 100
)
flight_status_summary = flight_status_summary.sort_values("total_flights", ascending=False)

flight_status_summary.to_csv("flight_status_summary.csv", index=False)
flight_status_summary

,status,total_flights,pct_of_total
0,Arrived,16707,50.442318
5,Scheduled,15383,46.444854
4,On Time,518,1.563962
1,Cancelled,414,1.249962
3,Departed,58,0.175115
2,Delayed,41,0.123789
